In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

In [0]:
import json
from datetime import datetime, timezone

import joblib
import numpy as np
import sklearn
import mlflow
from mlflow import MlflowClient

from churn.config import MODEL_NAME, GOLD_TABLE, SILVER_TABLE, ID, TARGET, UMBRAL, EXPORTS_VOLUME

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

In [0]:
champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
uri_modelo = f"models:/{MODEL_NAME}/{champion.version}"

pipe = mlflow.sklearn.load_model(uri_modelo)

In [0]:
ruta_modelo = f"{EXPORTS_VOLUME}/churn_classifier.joblib"
joblib.dump(pipe, ruta_modelo)

metadatos = {
    "model_name": MODEL_NAME,
    "version": int(champion.version),
    "run_id": champion.run_id,
    "umbral": UMBRAL,
    "scikit_learn": sklearn.__version__,
    "exportado": datetime.now(timezone.utc).isoformat(),
}

with open(f"{EXPORTS_VOLUME}/churn_classifier.json", "w") as f:
    json.dump(metadatos, f, indent=2)

In [0]:
cargado = joblib.load(ruta_modelo)

X = spark.table(GOLD_TABLE).toPandas().drop(columns=[ID, TARGET]).head(200)

assert np.allclose(cargado.predict_proba(X), pipe.predict_proba(X)), "El modelo exportado no predice igual"

In [0]:
from churn.config import SILVER_TABLE

clientes = spark.table(SILVER_TABLE).toPandas()
actuales = clientes[clientes[TARGET] == 0].drop(columns=[TARGET])

assert len(actuales) == 5174, f"{len(actuales)} clientes actuales"

actuales.to_csv(f"{EXPORTS_VOLUME}/clientes_actuales.csv", index=False)